In [1]:
from pathlib import Path
import torch
import numpy as np
import matplotlib.pyplot as plt
import json
from itertools import combinations
from scipy.stats import ttest_ind
import pandas as pd

Entire the desired output_directory, model_name, and metric we care about. Then run all the code below

In [2]:
output_dir = 'stack'

output_dir = Path('/data/vision/polina/users/marcusbl/bin_class/outputs') / output_dir

model_name = 'model_loss'
metrics = ['auc']

Parse results into Pandas DF

In [3]:
results = []

for test_dir in output_dir.iterdir():
    for run_dir in test_dir.iterdir():
        if not run_dir.is_dir():
            continue

        with open(run_dir / 'test_info' / 'metric_info.json') as f:
            metric_info = json.load(f)

        flat = {
            (model, metric): value
            for model, metrics in metric_info.items()
            for metric, value in metrics.items()
        }

        results.append({
            "test": test_dir.name,
            "run": run_dir.name,
            **flat
        })

df = pd.DataFrame(results).set_index(["test", "run"])

df.columns = pd.MultiIndex.from_tuples(df.columns)
df = df.sort_index(axis=1).sort_index()


In [4]:
df['model_auc']

acc    auc  epoch     f1  fn   fp    fpr   loss   prec  recall  \
test   run                                                                      
none   run0  0.871  0.909     -1  0.594  67   34  0.053  0.281  0.685   0.525   
       run1  0.912  0.959     -1  0.735  21   49  0.072  0.204  0.664   0.822   
       run2  0.911  0.953     -1  0.764  40   33  0.050  0.236  0.781   0.747   
       run3  0.853  0.944     -1  0.710  20   88  0.151  0.322  0.600   0.868   
       run4  0.929  0.940     -1  0.763  50    4  0.006  0.294  0.956   0.635   
       run5  0.899  0.975     -1  0.754   9   61  0.106  0.234  0.637   0.922   
       run6  0.835  0.919     -1  0.615  25  130  0.164  0.369  0.488   0.832   
       run7  0.888  0.943     -1  0.657  19   76  0.103  0.266  0.545   0.827   
       run8  0.873  0.916     -1  0.548  53   41  0.065  0.326  0.582   0.518   
       run9  0.929  0.958     -1  0.800  17   42  0.061  0.198  0.738   0.874   
stack2 run0  0.877  0.924     -1  0.694  32   64  0.100  0.294  0.630   0.773   
       run1  0.879  0.963     -1  0.688  12   84  0.124  0.247  0.558   0.898   
       run2  0.901  0.966     -1  0.777  17   64  0.097  0.228  0.688   0.892   
       run3  0.844  0.931     -1  0.695  22   92  0.158  0.371  0.586   0.855   
       run4  0.923  0.945     -1  0.746  52    6  0.010  0.269  0.934   0.620   
       run5  0.919  0.964     -1  0.770  22   34  0.059  0.192  0.734   0.810   
       run6  0.871  0.926     -1  0.661  31   90  0.114  0.338  0.567   0.792   
       run7  0.903  0.959     -1  0.699  15   67  0.091  0.209  0.586   0.864   
       run8  0.874  0.911     -1  0.583  45   48  0.076  0.320  0.575   0.591   
       run9  0.930  0.976     -1  0.805  15   43  0.062  0.183  0.736   0.889   
stack3 run0  0.891  0.927     -1  0.674  53   32  0.050  0.287  0.733   0.624   
       run1  0.950  0.957     -1  0.817  29   11  0.016  0.159  0.890   0.754   
       run2  0.894  0.950     -1  0.752  26   61  0.092  0.249  0.684   0.835   
       run3  0.873  0.943     -1  0.734  24   69  0.119  0.298  0.650   0.842   
       run4  0.918  0.943     -1  0.785  24   38  0.061  0.218  0.748   0.825   
       run5  0.865  0.964     -1  0.691  11   83  0.144  0.323  0.559   0.905   
       run6  0.832  0.930     -1  0.629  15  143  0.181  0.410  0.484   0.899   
       run7  0.901  0.950     -1  0.674  23   61  0.083  0.211  0.588   0.791   
       run8  0.881  0.920     -1  0.604  43   45  0.072  0.282  0.598   0.609   
       run9  0.933  0.963     -1  0.803  23   32  0.046  0.189  0.778   0.830   

              tn   tp    tpr  
test   run                    
none   run0  605   74  0.525  
       run1  627   97  0.822  
       run2  629  118  0.747  
       run3  493  132  0.868  
       run4  616   87  0.635  
       run5  517  107  0.922  
       run6  661  124  0.832  
       run7  662   91  0.827  
       run8  587   57  0.518  
       run9  650  118  0.874  
stack2 run0  575  109  0.773  
       run1  592  106  0.898  
       run2  598  141  0.892  
       run3  489  130  0.855  
       run4  614   85  0.620  
       run5  544   94  0.810  
       run6  701  118  0.792  
       run7  671   95  0.864  
       run8  580   65  0.591  
       run9  649  120  0.889  
stack3 run0  607   88  0.624  
       run1  665   89  0.754  
       run2  601  132  0.835  
       run3  512  128  0.842  
       run4  582  113  0.825  
       run5  495  105  0.905  
       run6  648  134  0.899  
       run7  677   87  0.791  
       run8  583   67  0.609  
       run9  660  112  0.830

Average/Variance for desired metrics over all runs for the desired model

In [5]:
m = df[model_name]

stats = m.groupby(level='test').agg(['mean', 'var'])

stats = stats.sort_values(by=('auc', 'mean'), ascending=False)

stats[[(metric, stat) for metric in metrics for stat in ['mean', 'var']]]

auc          
          mean       var
test                    
stack2  0.9433  0.000374
stack3  0.9399  0.000381
none    0.9388  0.000492

Paired T-test for signficance. Note that it's paired b/c same seed for each 

In [6]:
from itertools import combinations
from scipy.stats import ttest_rel

rows = []

tests = m.index.get_level_values("test").unique()

for metric in metrics:
    for test_a, test_b in combinations(tests, 2):
        a = m.loc[test_a, metric].dropna()
        b = m.loc[test_b, metric].dropna()

        # align by run name so only matching seeds are compared
        paired = pd.concat(
            [a.rename("a"), b.rename("b")],
            axis=1,
            join="inner"
        ).dropna()

        if len(paired) < 2:
            t_stat = np.nan
            p_value = np.nan
        else:
            t_stat, p_value = ttest_rel(paired["a"], paired["b"])

        rows.append({
            "metric": metric,
            "test_a": test_a,
            "test_b": test_b,
            "mean_a": paired["a"].mean() if len(paired) else np.nan,
            "mean_b": paired["b"].mean() if len(paired) else np.nan,
            "diff": (paired["a"] - paired["b"]).mean() if len(paired) else np.nan,
            "n_pairs": len(paired),
            "t_stat": t_stat,
            "p_value": p_value,
        })

paired_tstats = (
    pd.DataFrame(rows)
    .set_index(["metric", "test_a", "test_b"])
    # .sort_values(["metric", "p_value"])
)

paired_tstats

mean_a  mean_b    diff  n_pairs    t_stat   p_value
metric test_a test_b                                                     
auc    none   stack2  0.9388  0.9433 -0.0045       10 -1.214050  0.255623
              stack3  0.9388  0.9399 -0.0011       10 -0.361591  0.726001
       stack2 stack3  0.9433  0.9399  0.0034       10  1.122031  0.290891